In [31]:
import IPython
import torch
from effortless_config import Config
import numpy as np
import yaml

from ddsp.core import mean_std_loudness

class args(Config):
    CONFIG = "config.yaml"

with open(args.CONFIG, "r") as config_file:
    config = yaml.safe_load(config_file)

# Artificial envelope

In [9]:
loud = np.load("preprocessed/vn/loudness.npy")
print(loud.min(), loud.max(), loud.mean())

-16.713612 -3.043796 -6.4801626


In [10]:
def make_envelope(frames_per_note, n_notes, attack=5, decay=15):
    env = np.full(frames_per_note, -6.5, dtype=np.float32)   # sustain at mean
    env[:attack]  = np.linspace(-16.0, -4.0, attack)         # attack
    env[-decay:]  = np.linspace(-6.5,  -16.0, decay)         # decay
    return np.tile(env, n_notes)

loudness_vals = make_envelope(frames_per_note, len(notes_hz))
loudness = torch.tensor(loudness_vals).reshape(1, total_frames, 1)

In [14]:
model_violin_full = torch.jit.load("export/runs_test_full_20260604_194722_vn/ddsp_vn_pretrained.ts")
model_violin_test = torch.jit.load("export/runs_test_20260604_142130_vn/ddsp_vn_pretrained.ts")
model_flute = torch.jit.load("export/runs_test_20260604_142130_fl/ddsp_fl_pretrained.ts")

sr = config["preprocess"]["sampling_rate"]
block_size = config["preprocess"]["block_size"]
secs_per_note = 1

notes_hz = [261.63, 293.66, 329.63, 349.23, 392.00, 440.00, 493.88, 523.25]
frames_per_note = int(secs_per_note * sr / block_size)
total_frames = frames_per_note * len(notes_hz)

pitch_vals = np.repeat(notes_hz, secs_per_note * sr / block_size)

pitch = torch.tensor(pitch_vals, dtype=torch.float32).reshape(1, total_frames, 1)

audio_violin_full = model_violin_full(pitch, loudness).squeeze().detach().cpu().numpy()
audio_violin_test = model_violin_test(pitch, loudness).squeeze().detach().cpu().numpy()
audio_flute = model_flute(pitch, loudness).squeeze().detach().cpu().numpy()

In [18]:
 IPython.display.Audio(audio_flute, rate=config["preprocess"]["sampling_rate"])

In [16]:
 IPython.display.Audio(audio_violin_full, rate=config["preprocess"]["sampling_rate"])

In [17]:
 IPython.display.Audio(audio_violin_test, rate=config["preprocess"]["sampling_rate"])

# Timbre transfer

In [27]:
input_path = "../data/URMP/Dataset/01_Jupiter_vn_vc/AuSep_1_vn_01_Jupiter.wav"

In [21]:
from preprocess import preprocess

In [22]:
with open(args.CONFIG, "r") as config:
    config = yaml.safe_load(config)

In [28]:
x, p, l = preprocess(input_path, **config["preprocess"])

I0000 00:00:1780660346.321942    6907 service.cc:145] XLA service 0x7f8924004140 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780660346.322063    6907 service.cc:153]   StreamExecutor device (0): NVIDIA A40, Compute Capability 8.6


  8/501 ━━━━━━━━━━━━━━━━━━━━ 12s 26ms/step

I0000 00:00:1780660353.982740    6907 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


501/501 ━━━━━━━━━━━━━━━━━━━━ 27s 35ms/step


/home/jovyan/venv/lib/python3.11/site-packages/librosa/core/convert.py:1870: RuntimeWarning: divide by zero encountered in log10
  + 2 * np.log10(f_sq)


In [77]:
pitch = torch.from_numpy(p).float().view(1, -1, 1)
loudness = torch.from_numpy(l).float().view(1, -1, 1)
audio = model_violin_full(pitch, loudness).reshape(-1).detach().cpu().numpy()

In [78]:
 IPython.display.Audio(audio, rate=config["preprocess"]["sampling_rate"])

# Interpolate between Outputs

In [ ]:
def get_hidden(model, pitch, loudness):
    hidden = torch.cat([
            model.in_mlps[0](pitch),
            model.in_mlps[1](loudness),
        ], -1)
    hidden = torch.cat([model.gru(hidden)[0], pitch, loudness], -1)
    hidden = model.out_mlp(hidden)

    return hidden


def get_amplitudes(model, pitch, loudness):
    hidden = get_hidden(model, pitch, loudness)

    param = scale_function(model.proj_matrices[0](hidden))

    total_amp = param[..., :1]
    amplitudes = param[..., 1:]

    amplitudes = remove_above_nyquist(
        amplitudes,
        pitch,
        model.sampling_rate,
    )
    amplitudes /= amplitudes.sum(-1, keepdim=True)
    amplitudes *= total_amp

    amplitudes = upsample(amplitudes, model.block_size)

    return amplitudes

def get_filter_impulse(model, pitch, loudness):
    hidden = get_hidden(model, pitch, loudness)

    param = scale_function(model.proj_matrices[1](hidden) - 5)

    impulse = amp_to_impulse_response(param, model.block_size)

    return impulse


def interpolated_output(model1, model2, pitch, loudness, alpha):

    amplitudes1 = get_amplitudes(model1, pitch, loudness)
    amplitudes2 = get_amplitudes(model2, pitch, loudness)
    amplitudes = (1 - alpha) * amplitudes1 + alpha * amplitudes2

    pitch = upsample(pitch, model1.block_size)
    harmonic = harmonic_synth(pitch, amplitudes, model1.sampling_rate)

    impulse1 = get_filter_impulse(model1, pitch, loudness)
    impulse2 = get_filter_impulse(model2, pitch, loudness)
    impulse = (1 - alpha) * impulse1 + alpha * impulse2

    noise = torch.rand(
        impulse.shape[0],
        impulse.shape[1],
        model.block_size,
    ).to(impulse) * 2 - 1

    noise = fft_convolve(noise, impulse).contiguous()
    noise = noise.reshape(noise.shape[0], -1, 1)

    signal = harmonic + noise

    return signal